# Look-ahead · Process  `[EVAL]`

**What does each therapist behaviour *do to the patient*, and does look-ahead change it?** Every MI
instrument in the lake is a per-conversation aggregate. The `MIPROC` coder assigns one MITI/MISC-style
code to EVERY therapist utterance (OQ CQ SR CR AF PRA GI PERS SEEK CONF OTH) and one to EVERY patient
utterance (CT ST NEU), in order, so the classic MI process-research questions become answerable:

1. **Code mix** — the therapist's behaviour profile per arm × iteration (levels + persona-paired K contrast).
2. **Yield** — P(next patient utterance is change talk | therapist code): which behaviours *produce* change
   talk, and does K=5 change the yields?
3. **Responsiveness** — what the policy does after change talk vs after sustain talk (reflect it? praise it?).
4. **Within-session change talk** — trajectory by patient turn bin, time to first change talk.
5. **Parity** — MIPROC counts summed per conversation vs the same grader's MITI / PCT counts: the
   free validity check of the new coder.

**Conventions.** Sign `+ ⇒ K=0 higher`, pairing unit `persona_id`, both graders side by side and never
averaged (judge column, or one panel per grader). The scripted opener (therapist #1) is excluded from every
rate and transition. Holm scope is stated per table. `BOOT_SEED` everywhere.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 60, "display.max_colwidth", 120)

import eda_analysis
from eda_analysis import exports, plotting, stats, behavior, lookahead, reliability, process
from eda_analysis.constants import BOOT_SEED, set_active_judge, judge_dirname, PRIMARY_JUDGE_TAG

cfg = eda_analysis.EdaConfig(family="lookahead/process", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp figures/_provenance.md (reset just removed the one notebook_setup wrote)

## 0 · Load the MIPROC codes under every grader that has them

In [ ]:
KA = eda_analysis.cross_k_arms(S)
K_ARMS = [a.label for a in KA if a.label in ("PTO_LA0", "PTO_LA5", "GRPO_LA0", "GRPO_LA5")]
PAL = plotting.arm_palette(K_ARMS)

def support(frame, **kw) -> str:
    note = eda_analysis.support_note(frame, **kw)
    return f" {note}" if note else ""

TAGS = {judge_dirname(t): t for t in reliability.second_judge_tags()}
TAGS[judge_dirname(PRIMARY_JUDGE_TAG)] = ""
TAGS = dict(sorted(TAGS.items(), key=lambda kv: kv[1] != ""))     # primary first
CONV, MITI, PCT = {}, {}, {}
try:
    for j, tag in TAGS.items():
        set_active_judge(tag, 0)
        c = process.load_miproc(KA)
        if c.empty:
            print(f"{j}: no MIPROC scores on disk — skipped"); continue
        CONV[j] = c
        MITI[j] = behavior.load_miti_behavior(KA, attach_persona=False)
        PCT[j] = behavior.load_pct_behavior(KA, attach_persona=False)
        cov = c.groupby(["arm", "iteration"]).size()
        print(f"{j}: {len(c):,} conversations | arms {sorted(c.arm.unique())} | "
              + ", ".join(f"{a}:{cov[a].index.min()}..{cov[a].index.max()} ({cov[a].min()}-{cov[a].max()}/state)" for a in sorted(c.arm.unique())))
finally:
    set_active_judge("", 0)
JUDGES = list(CONV)
assert JUDGES, "lookahead/process needs MIPROC scores under at least one grader"
ARMS_SCORED = sorted(set().union(*[set(c.arm.unique()) for c in CONV.values()]))
K_ARMS = [a for a in K_ARMS if a in ARMS_SCORED]

## 1 · Code mix — levels and the persona-paired K contrast

Every per-conversation process metric (therapist code shares, technique ratios, patient change talk,
contingencies) as a level table per grader, then K=0 − K=5 paired on persona at every matched iteration.
Holm across ITERATIONS within (grader, method, metric).

In [ ]:
LEVELS, KT_ALL, KTS_ALL = {}, [], []
for j in JUDGES:
    LEVELS[j] = process.state_table(CONV[j])
    SL = process.to_scores_long(CONV[j], KA)
    KT = lookahead.paired_k_frames({j: SL}, metrics=process.PROCESS_K_METRICS, holm_family="iterations")
    if not KT.empty:
        KT["lower_better"] = KT["metric"].isin(process.LOWER_BETTER)
        KT_ALL.append(KT); KTS_ALL.append(lookahead.k_summary(KT).assign(judge=j))
    exports.save_table(LEVELS[j].round(4), f"process_levels_{j}", caption=(
        f"[{j}] Per (arm, iteration): mean and SE over the state's conversations of every per-conversation process "
        "metric — " + "; ".join(f"{m} = {d}" for m, d in process.PROCESS_METRIC_LABELS.items()) + ". Opener excluded."
        + support(LEVELS[j])))
    display(LEVELS[j][["arm", "iteration"] + [f"th_{c}_rate" for c in process.TH_CODES] + ["ct_prop", "reached_ct", "ct_after_q", "refl_after_ct"]].round(3))
KT = pd.concat(KT_ALL, ignore_index=True) if KT_ALL else pd.DataFrame()
KTS = pd.concat(KTS_ALL, ignore_index=True) if KTS_ALL else pd.DataFrame()
display(KTS)
exports.save_table(KT.round(4), "k_process_paired", caption=(
    "Persona-paired K=0 − K=5 on every per-conversation process metric, per grader, method and matched iteration. "
    f"{lookahead.SIGN_NOTE} Holm across ITERATIONS within (judge, method, metric). lower_better marks the "
    "MI-inconsistent codes, closed questions, sustain talk, time-to-first-change-talk, praise-after-sustain-talk and "
    "question chains." + support(KT, subject="no later matched iteration")))
exports.save_table(KTS, "k_process_summary", caption=(
    "Per (grader, method, metric): tally of matched iterations where K=0 − K=5 is significant after Holm, by sign, "
    "with the mean dz."))
for j in JUDGES:
    fig = plotting.k_text_forest(KT[KT.judge == j], lower_better=process.LOWER_BETTER,
                                 labels=process.PROCESS_METRIC_LABELS)
    exports.save_fig(fig, f"k_process_forest_{j}", caption=(
        f"[{j}] Persona-paired dz of K=0 − K=5 on every process metric at each method's last matched iteration; "
        "lower-better metrics sign-flipped so a bar to the right always reads 'K=0 better'. Holm stars across "
        "iterations within (method, metric)."))
    plt.show()
    fig = plotting.code_mix_fig(LEVELS[j], process.TH_CODES, arms=K_ARMS)
    exports.save_fig(fig, f"code_mix_{j}", caption=(
        f"[{j}] The therapist's MIPROC code mix by iteration (stacked shares of policy turns), one panel per arm."
        + support(LEVELS[j])))
    plt.show()

## 2 · Yield — what each therapist behaviour produces in the patient

P(next patient utterance = CT | therapist code), per state, Wilson 95 % intervals. The MI claim is that
reflections and open questions elicit change talk while persuasion and praise do not; the look-ahead claim
is that K=5 should select behaviours whose payoff arrives in the next turns.

In [ ]:
YIELD, RESP = {}, {}
for j in JUDGES:
    YIELD[j] = process.transition_yield(CONV[j])
    RESP[j] = process.responsiveness(CONV[j])
    exports.save_table(YIELD[j].round(4), f"yield_{j}", caption=(
        f"[{j}] Per (arm, iteration, therapist code): n policy turns with that code and the share of the patient's "
        "next utterance coded CT / ST / NEU; p_ct_lo/hi = Wilson 95% interval on the change-talk yield. Opener pairs "
        "excluded." + support(YIELD[j])))
    exports.save_table(RESP[j].round(4), f"responsiveness_{j}", caption=(
        f"[{j}] Per (arm, iteration, preceding patient code): the distribution of the therapist's next code (share "
        "per code, plus p_reflect = SR+CR and p_question = OQ+CQ). What the policy does after change talk vs sustain talk."
        + support(RESP[j])))
    last = YIELD[j][YIELD[j].iteration == YIELD[j].groupby("arm").iteration.transform("max")]
    display(last.pivot_table(index="th_code", columns="arm", values="p_ct").reindex(process.TH_CODES).round(3))
fig = plotting.yield_fig(YIELD, process.TH_CODES, arms=K_ARMS, palette=PAL)
exports.save_fig(fig, "yield", caption=(
    "P(next patient utterance = change talk | therapist code) at each arm's endpoint beside the pooled base, Wilson "
    "95% intervals, one panel per grader; codes with fewer than 20 turns in a state are left blank."))
plt.show()
MATS = {}
for j in JUDGES:
    for a in K_ARMS:
        it = int(CONV[j][CONV[j].arm == a].iteration.max())
        MATS[f"{a} @ it {it} — {j}"] = process.transition_matrix(CONV[j], a, it)
fig = plotting.transition_heatmap(MATS, title="Therapist code → next patient code (row-normalised), endpoints")
exports.save_fig(fig, "transition_endpoints", caption=(
    "Row-normalised therapist-code × next-patient-code contingency at each arm's endpoint, per grader; n per row on "
    "the y labels. The yield table in matrix form."))
plt.show()

## 3 · Within-session change talk and time to first change talk

In [ ]:
TRAJ = {}
for j in JUDGES:
    TRAJ[j] = process.ct_trajectory(CONV[j])
    exports.save_table(TRAJ[j].round(4), f"ct_trajectory_{j}", caption=(
        f"[{j}] Per (arm, iteration, patient turn bin): change-talk and sustain-talk share of patient utterances, n "
        "utterances, and the share of the state's conversations reaching the bin." + support(TRAJ[j])))
fig = plotting.ct_trajectory_fig(TRAJ, arms=K_ARMS, palette=PAL)
exports.save_fig(fig, "ct_trajectory", caption=(
    "Change-talk share by patient turn bin: each arm at its endpoint (K=0 solid, K=5 dashed) against the pooled base, "
    "one panel per grader."))
plt.show()
TTF = pd.concat([LEVELS[j][["arm", "iteration", "reached_ct", "reached_ct_se", "first_ct_pos", "first_ct_pos_se", "n"]].assign(judge=j)
                 for j in JUDGES], ignore_index=True)
display(TTF[TTF.iteration.isin([0, TTF.iteration.max()])].round(3))
exports.save_table(TTF.round(4), "time_to_change_talk", caption=(
    "Per (grader, arm, iteration): share of conversations that reach any change talk, and among those the mean "
    "patient turn index of the first change talk (lower = earlier). The K contrast on both is in k_process_paired."))

## 4 · Parity with the conversation-level instruments (validity of the coder)

MIPROC's per-utterance codes summed per conversation, against the SAME grader's MITI behaviour counts
(questions = OQ+CQ, SR, CR, AF, GI, Persuade, Seek) and PCT counts (CT, ST, NEU). Spearman ρ across
conversations within each state, Fisher-z pooled; and the mean absolute count difference.

In [ ]:
PAR, PARP = {}, {}
for j in JUDGES:
    PAR[j] = process.parity(CONV[j], MITI.get(j), PCT.get(j))
    PARP[j] = process.parity_pooled(PAR[j]) if not PAR[j].empty else pd.DataFrame()
    exports.save_table(PAR[j].round(4), f"parity_{j}", caption=(
        f"[{j}] Per (arm, iteration, instrument count): Spearman ρ across conversations between the MIPROC count "
        "(codes summed per conversation, opener included as in MITI) and the grader's own MITI / PCT count, the two "
        "means and the mean absolute difference."))
    exports.save_table(PARP[j].round(4), f"parity_pooled_{j}", caption=(
        f"[{j}] Fisher-z pooled within-state ρ and n-weighted mean |Δ| per instrument count — the coder's agreement "
        "with the conversation-level instruments it should reproduce."))
    display(PARP[j].round(3))

In [ ]:
NUM = process.process_numbers(LEVELS, YIELD, PARP)
exports.save_numbers("process_numbers", NUM, caption=(
    "Ledger of the family's endpoint scalars per grader (code shares, change talk, yields, parity ρ), each citing "
    "its source table."))
print(f"{len(NUM)} ledger keys")

In [ ]:
exports.prune_orphan_captions(); exports.build_index()